# 10.1 指标贡献

给 7.1 的候选模型一个能讲给人听的解释：预测主要靠哪几个理化指标，和 2.1 看到的相关关系对不对得上。
只在验证集上算，最终评估集不碰；模型不重训。解释的是模型的行为，不是酒好坏的原因。


In [1]:
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
STEP = ROOT / "steps/10_可解释性/10.1_指标贡献"
OUT = STEP / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
S21 = ROOT / "steps/02_EDA/2.1_理化指标与quality的探索分析/outputs"
S31 = ROOT / "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs"
S71 = ROOT / "steps/07_模型选择与训练/7.1_候选模型比较/outputs"
FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
            "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]
WINES = {"red": "红酒", "white": "白酒"}
CANDIDATES = ("Ridge", "随机森林")
SEED = 20260914
REPEATS = 20

run = dsflow.start_run("10.1", project=ROOT, hypothesis="候选模型主要靠 alcohol 等少数几个指标，且方向与 2.1 的相关关系一致")
raw = {}
for w, name in WINES.items():
    path = ROOT / f"data/winequality-{w}.csv"
    run.log_input(path, name=name)
    raw[w] = pd.read_csv(path, sep=";").assign(source_row=lambda d: np.arange(1, len(d) + 1))
run.log_input(S31 / "split_assignments.csv", name="划分表")
splits = pd.read_csv(S31 / "split_assignments.csv")
eda = json.loads((S21 / "eda.json").read_text(encoding="utf-8"))
models = {(w, c): joblib.load(S71 / "models" / f"{w}_{c}.joblib") for w in WINES for c in CANDIDATES}
INPUT_FILES = ["data/winequality-red.csv", "data/winequality-white.csv", "steps/02_EDA/2.1_理化指标与quality的探索分析/outputs/eda.json",
               "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs/split_assignments.csv",
               *[f"steps/07_模型选择与训练/7.1_候选模型比较/outputs/models/{w}_{c}.joblib" for w in WINES for c in CANDIDATES]]
fingerprint = lambda rel: {"bytes": (ROOT / rel).stat().st_size, "sha256": hashlib.sha256((ROOT / rel).read_bytes()).hexdigest()}
inputs_before = {rel: fingerprint(rel) for rel in INPUT_FILES}
run.log_params({"方法": "置换重要性：在验证行上打乱一个指标列，看 MAE 变差多少", "重复": REPEATS, "种子": SEED, "生成器": "PCG64，每类酒每候选各一个",
                "顺序": "按 11 个指标的列顺序，每个指标连续打乱 20 次"})
valid = {}
for w in WINES:
    df = raw[w].merge(splits[splits["wine"] == w].drop(columns="wine"), on="source_row", how="left", validate="one_to_one")
    valid[w] = df[df["split"] == "validation"].sort_values("source_row").reset_index(drop=True)
print("验证行：红酒 %d、白酒 %d；最终评估行不碰" % (len(valid["red"]), len(valid["white"])))


验证行：红酒 331、白酒 983；最终评估行不碰


In [2]:
def predict(m, X):
    return m["model"].predict((X - m["scaler"]["mean"]) / m["scaler"]["std"]) if "scaler" in m else m["model"].predict(X)

rows = []
for w, name in WINES.items():
    X = valid[w][FEATURES].to_numpy(float)
    y = valid[w]["quality"].to_numpy(float)
    for cand in CANDIDATES:
        m = models[(w, cand)]
        base_mae = float(np.abs(y - predict(m, X)).mean())
        rng = np.random.Generator(np.random.PCG64(SEED))
        for j, feat in enumerate(FEATURES):
            worse = []
            for _ in range(REPEATS):
                Xp = X.copy()
                Xp[:, j] = X[rng.permutation(len(X)), j]
                worse.append(float(np.abs(y - predict(m, Xp)).mean()) - base_mae)
            rows.append({"wine": w, "candidate": cand, "feature": feat, "validation_mae": base_mae,
                         "mae_increase_when_shuffled": float(np.mean(worse)), "spearman_with_quality_2_1": eda["files"][w]["indicators"][feat]["spearman_quality"]})
imp = pd.DataFrame(rows)
for w in WINES:
    m = models[(w, "Ridge")]
    coef = dict(zip(FEATURES, m["model"].coef_.tolist()))
    imp.loc[imp["wine"] == w, "ridge_standardized_coef"] = imp.loc[imp["wine"] == w, "feature"].map(coef)
imp["rank_within_model"] = imp.groupby(["wine", "candidate"])["mae_increase_when_shuffled"].rank(ascending=False, method="first").astype(int)
imp = imp.sort_values(["wine", "candidate", "rank_within_model"]).reset_index(drop=True)
imp.to_csv(OUT / "importance.csv", index=False)
top = imp[imp["rank_within_model"] <= 3][["wine", "candidate", "rank_within_model", "feature", "mae_increase_when_shuffled", "spearman_with_quality_2_1", "ridge_standardized_coef"]]
print(top.round(4).to_string(index=False))


 wine candidate  rank_within_model              feature  mae_increase_when_shuffled  spearman_with_quality_2_1  ridge_standardized_coef
  red     Ridge                  1              alcohol                      0.1032                     0.4785                   0.2894
  red     Ridge                  2     volatile acidity                      0.0406                    -0.3806                  -0.1436
  red     Ridge                  3 total sulfur dioxide                      0.0262                    -0.1967                  -0.1384
  red      随机森林                  1              alcohol                      0.1161                     0.4785                   0.2894
  red      随机森林                  2            sulphates                      0.0513                     0.3771                   0.1553
  red      随机森林                  3     volatile acidity                      0.0264                    -0.3806                  -0.1436
white     Ridge                  1              

In [3]:
flags = []
for (w, cand), g in imp.groupby(["wine", "candidate"]):
    for _, r in g.iterrows():
        if cand == "Ridge" and np.sign(r["ridge_standardized_coef"]) != np.sign(r["spearman_with_quality_2_1"]) and abs(r["spearman_with_quality_2_1"]) >= 0.1:
            flags.append(f"{WINES[w]} Ridge：{r['feature']} 的系数符号（{r['ridge_standardized_coef']:+.3f}）与 2.1 的 Spearman（{r['spearman_with_quality_2_1']:+.3f}）相反")
        if r["rank_within_model"] <= 3 and abs(r["spearman_with_quality_2_1"]) < 0.1:
            flags.append(f"{WINES[w]} {cand}：{r['feature']} 排第 {int(r['rank_within_model'])}，但 2.1 里它和 quality 的 Spearman 只有 {r['spearman_with_quality_2_1']:+.3f}")
summary = {"method": {"permutation": f"验证行上打乱一列，{REPEATS} 次取平均的 MAE 增量", "seed": SEED, "generator": "PCG64"},
           "inputs_before": inputs_before, "inputs_after": {rel: fingerprint(rel) for rel in INPUT_FILES},
           "density_alcohol_spearman_2_1": {w: eda["files"][w]["inter_indicator_spearman"]["density"]["alcohol"] for w in WINES},
           "top3": {f"{WINES[w]}_{c}": g.sort_values("rank_within_model")["feature"].head(3).tolist() for (w, c), g in imp.groupby(["wine", "candidate"])},
           "flags": flags, "rows": imp.to_dict(orient="records")}
assert summary["inputs_after"] == inputs_before
(OUT / "importance.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")
run.log_artifact(OUT / "importance.csv", purpose="每类酒每候选每指标的置换重要性、Ridge 标准化系数、2.1 的 Spearman", kind="table")
run.log_artifact(OUT / "importance.json", purpose="同上，外加前三名、需要留意的指标、density–alcohol 相关", kind="table")
for w in WINES:
    for cand in CANDIDATES:
        g = imp[(imp["wine"] == w) & (imp["candidate"] == cand)].sort_values("rank_within_model")
        run.log_metrics({f"{WINES[w]}_{cand}_首位指标_MAE增量": float(g.iloc[0]["mae_increase_when_shuffled"])})
print("每个模型最依赖的三个指标：")
for k, v in summary["top3"].items():
    print(f"  {k}：{'、'.join(v)}")
print("需要留意：" if flags else "没有符号相反或相关接近零却排前三的指标。")
for f in flags:
    print("  -", f)
print("2.1 里 density 与 alcohol 的 Spearman：", {WINES[w]: round(v, 4) for w, v in summary["density_alcohol_spearman_2_1"].items()})


每个模型最依赖的三个指标：
  红酒_Ridge：alcohol、volatile acidity、total sulfur dioxide
  红酒_随机森林：alcohol、sulphates、volatile acidity
  白酒_Ridge：density、residual sugar、alcohol
  白酒_随机森林：alcohol、volatile acidity、free sulfur dioxide
需要留意：
  - 白酒 Ridge：residual sugar 排第 2，但 2.1 里它和 quality 的 Spearman 只有 -0.082
  - 白酒 随机森林：free sulfur dioxide 排第 3，但 2.1 里它和 quality 的 Spearman 只有 +0.024
2.1 里 density 与 alcohol 的 Spearman： {'红酒': -0.4624, '白酒': -0.8219}


In [4]:
conclusion = "；".join(f"{k} 最依赖 {'、'.join(v)}" for k, v in summary["top3"].items())
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


红酒_Ridge 最依赖 alcohol、volatile acidity、total sulfur dioxide；红酒_随机森林 最依赖 alcohol、sulphates、volatile acidity；白酒_Ridge 最依赖 density、residual sugar、alcohol；白酒_随机森林 最依赖 alcohol、volatile acidity、free sulfur dioxide
